# Module 25 — Text Preprocessing: Tokenization, Stemming & Stopwords

> **Track 3 · Yandex Big Tech & Search**  
> **Difficulty**: Intermediate  
> **Runtime**: ~5 minutes  
> **Key Libraries**: NLTK, Pandas, NumPy, Scikit-learn  
> **Prerequisite**: Module 1 (Optimized Pandas)

## Table of Contents

1. [Business Context](#1-business-context)
2. [Setup & Imports](#2-setup--imports)
3. [Synthetic Text Data](#3-synthetic-text-data)
4. [Text Cleaning](#4-text-cleaning)
5. [Tokenization](#5-tokenization)
6. [Stopword Removal](#6-stopword-removal)
7. [Stemming & Lemmatization](#7-stemming--lemmatization)
8. [Visualization](#8-visualization)
9. [Key Business Takeaway](#9-key-business-takeaway)

---
## §1 · Business Context

### Why text preprocessing matters for search?

Text preprocessing is **critical** for search and NLP tasks:
- **Normalization**: Convert text to consistent format.
- **Noise removal**: Eliminate irrelevant characters and words.
- **Feature extraction**: Prepare text for ML models.

**Applications at Yandex**:
1. **Search queries**: Preprocess user queries for matching.
2. **Document indexing**: Process web pages for search index.
3. **Recommendation systems**: Analyze user reviews and feedback.

**Key techniques**:
- **Tokenization**: Split text into words or subwords.
- **Stopword removal**: Remove common words (the, is, at).
- **Stemming/Lemmatization**: Reduce words to base form.

In this notebook we will:
1. Generate synthetic product descriptions.
2. Clean and normalize text.
3. Tokenize and remove stopwords.
4. Apply stemming and lemmatization.

---
## §2 · Setup & Imports

In [ ]:
# ── Reproducibility ──────────────────────────────────────────────
import numpy as np
import pandas as pd
import re
import warnings

np.random.seed(42)
warnings.filterwarnings("ignore")

# ── NLP libraries ────────────────────────────────────────────────
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer, WordNetLemmatizer

# Download required NLTK data
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)

# ── Visualization ────────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px

# ── Display settings ─────────────────────────────────────────────
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 100)
sns.set_theme(style="whitegrid")

print("✓ All imports loaded")

---
## §3 · Synthetic Text Data

In [ ]:
# Generate synthetic product descriptions
N_PRODUCTS = 1000

# Product categories and templates
categories = ['electronics', 'clothing', 'books', 'home', 'sports']
templates = [
    "The {adjective} {product} is perfect for {use_case}. It features {feature1} and {feature2}.",
    "This {adjective} {product} offers {feature1} with {feature2}. Ideal for {use_case}.",
    "Experience the best {product} with {feature1} and {feature2}. Great for {use_case}.",
]

adjectives = ['premium', 'high-quality', 'affordable', 'luxury', 'professional']
products = ['laptop', 'headphones', 'camera', 'smartphone', 'tablet']
features = ['fast processing', 'long battery life', 'crystal clear display', 'advanced AI', 'wireless connectivity']
use_cases = ['work', 'entertainment', 'gaming', 'photography', 'everyday use']

# Generate descriptions
descriptions = []
for i in range(N_PRODUCTS):
    template = np.random.choice(templates)
    desc = template.format(
        adjective=np.random.choice(adjectives),
        product=np.random.choice(products),
        feature1=np.random.choice(features),
        feature2=np.random.choice(features),
        use_case=np.random.choice(use_cases)
    )
    descriptions.append(desc)

df = pd.DataFrame({
    'product_id': range(N_PRODUCTS),
    'category': np.random.choice(categories, N_PRODUCTS),
    'description': descriptions
})

print(f"✓ Generated {N_PRODUCTS} product descriptions")
print(f"\nSample descriptions:")
for i in range(3):
    print(f"  {i+1}. {df['description'].iloc[i][:100]}...")

---
## §4 · Text Cleaning

In [ ]:
# Clean text
def clean_text(text):
    """Clean and normalize text."""
    # Convert to lowercase
    text = text.lower()
    
    # Remove special characters and digits
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

# Apply cleaning
df['cleaned'] = df['description'].apply(clean_text)

print("✓ Text cleaned")
print(f"\nSample cleaned text:")
for i in range(3):
    print(f"  Original: {df['description'].iloc[i][:80]}...")
    print(f"  Cleaned:  {df['cleaned'].iloc[i][:80]}...")
    print()

---
## §5 · Tokenization

In [ ]:
# Tokenize text
df['tokens'] = df['cleaned'].apply(word_tokenize)

# Statistics
df['token_count'] = df['tokens'].apply(len)

print("✓ Tokenization complete")
print(f"\nToken statistics:")
print(f"  Average tokens per description: {df['token_count'].mean():.1f}")
print(f"  Min tokens: {df['token_count'].min()}")
print(f"  Max tokens: {df['token_count'].max()}")
print(f"\nSample tokens:")
print(f"  {df['tokens'].iloc[0][:10]}...")

---
## §6 · Stopword Removal

In [ ]:
# Remove stopwords
stop_words = set(stopwords.words('english'))

def remove_stopwords(tokens):
    """Remove stopwords from token list."""
    return [token for token in tokens if token not in stop_words]

df['tokens_no_stop'] = df['tokens'].apply(remove_stopwords)
df['token_count_no_stop'] = df['tokens_no_stop'].apply(len)

print("✓ Stopwords removed")
print(f"\nImpact of stopword removal:")
print(f"  Average tokens before: {df['token_count'].mean():.1f}")
print(f"  Average tokens after:  {df['token_count_no_stop'].mean():.1f}")
print(f"  Reduction: {(1 - df['token_count_no_stop'].mean() / df['token_count'].mean()) * 100:.1f}%")
print(f"\nSample tokens without stopwords:")
print(f"  {df['tokens_no_stop'].iloc[0][:10]}...")

---
## §7 · Stemming & Lemmatization

In [ ]:
# Apply stemming
stemmer = PorterStemmer()

def stem_tokens(tokens):
    """Apply stemming to tokens."""
    return [stemmer.stem(token) for token in tokens]

df['tokens_stemmed'] = df['tokens_no_stop'].apply(stem_tokens)

# Apply lemmatization
lemmatizer = WordNetLemmatizer()

def lemmatize_tokens(tokens):
    """Apply lemmatization to tokens."""
    return [lemmatizer.lemmatize(token) for token in tokens]

df['tokens_lemmatized'] = df['tokens_no_stop'].apply(lemmatize_tokens)

print("✓ Stemming and lemmatization complete")
print(f"\nSample comparison:")
print(f"  Original:    {df['tokens_no_stop'].iloc[0][:5]}")
print(f"  Stemmed:     {df['tokens_stemmed'].iloc[0][:5]}")
print(f"  Lemmatized:  {df['tokens_lemmatized'].iloc[0][:5]}")

---
## §8 · Visualization

In [ ]:
# Visualize token statistics
fig = make_subplots(rows=1, cols=2, subplot_titles=['Token Count Distribution', 'Top Words After Processing'])

# Token count distribution
fig.add_trace(go.Histogram(
    x=df['token_count_no_stop'],
    nbinsx=30,
    name='Token Count'
), row=1, col=1)

# Top words
all_words = [word for tokens in df['tokens_lemmatized'] for word in tokens]
word_freq = pd.Series(all_words).value_counts().head(20)

fig.add_trace(go.Bar(
    x=word_freq.values,
    y=word_freq.index,
    orientation='h',
    name='Word Frequency'
), row=1, col=2)

fig.update_layout(height=400, showlegend=True)
fig.show()

print("💡 Key insights:")
print("   - Most descriptions have 10-20 meaningful tokens")
print("   - Product-related words dominate after stopword removal")
print("   - Lemmatization preserves word meaning better than stemming")

---
## §9 · Key Business Takeaway

> ### 🎯 Action Items for Search & NLP Teams
>
> | Aspect | Recommendation |
> |--------|---------------|
> | **When to use** | Search indexing, query processing, text classification |
> | **Cleaning** | Lowercase, remove special chars, normalize whitespace |
> | **Tokenization** | Use NLTK or spaCy for robust tokenization |
> | **Stopwords** | Remove domain-specific stopwords |
> | **Normalization** | Lemmatization > Stemming for search |
>
> ### 💡 Yandex-Specific Guidelines
>
> 1. **Search query preprocessing**:
>    ```
>    Query → Clean → Tokenize → Remove stopwords → Lemmatize → Match
>    ```
>
> 2. **Domain-specific preprocessing**:
>    | Domain | Special Handling |
>    |--------|------------------|
>    | E-commerce | Brand names, product codes |
>    | News | Named entities, dates |
>    | Social media | Emojis, hashtags, mentions |
>
> 3. **Multilingual support**:
>    - Use language-specific stopword lists
>    - Apply language-specific stemmers/lemmatizers
>    - Handle transliteration for Russian text
>
> ### 🔑 Key Takeaways
>
> 1. **Text preprocessing significantly impacts** downstream model performance.
> 2. **Lemmatization preserves meaning** better than stemming for search.
> 3. **Domain-specific stopwords** improve relevance.
> 4. **Consistent preprocessing** is essential for indexing and querying.
> 5. **Multilingual preprocessing** requires language-specific tools.